# 🎙️ Pipeline Fallback Step: Merging Speaker Turns: Pure Python List Merging

Refines diarized speaker timelines using a memory-efficient single-pass Python accumulator list loop.

## Environment Setup

In [ ]:
import json
import os

## Google Drive Mount & Form Configuration

In [ ]:
try:
    drive.mount('/content/drive')
    print("Google Drive successfully mounted.")
except Exception as e:
    print(f"Drive mount error: {e}")

# @markdown ### 📂 Refinement Configuration
input_json_path = "/content/drive/MyDrive/annam AI tasks/outreach activity/transcription result/backup fallback strategies/MarauliKhurad3/maraulikhurad3_raw_timeline.json" # @param {type:"string"}
output_json_path = "/content/drive/MyDrive/annam AI tasks/outreach activity/transcription result/backup fallback strategies/MarauliKhurad3/maraulikhurad3_refined_timeline.json" # @param {type:"string"}

os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

## Execute Fallback Processing

In [ ]:
# @markdown ### ⏱️ Refinement Parameters
max_merge_gap = 1.5 # @param {type:"number"}

try:
    with open(input_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    def parse_time(time_str, idx):
        part = time_str.split('-')[idx].replace('[','').replace(']','').strip()
        parts = part.split(':')
        return float(parts[-2])*60 + float(parts[-1])
        
    refined_entries = []
    if data:
        current_entry = data[0].copy()
        current_start = parse_time(current_entry['time'], 0)
        current_end = parse_time(current_entry['time'], 1)
        
        for next_entry in data[1:]:
            next_start = parse_time(next_entry['time'], 0)
            next_end = parse_time(next_entry['time'], 1)
            
            if next_entry['speaker'] == current_entry['speaker'] and (next_start - current_end) <= max_merge_gap:
                current_entry['text'] = current_entry['text'] + " " + next_entry['text']
                current_end = next_end
                current_entry['time'] = f"[{int(current_start//60):02d}:{int(current_start%60):02d} - {int(current_end//60):02d}:{int(current_end%60):02d}]"
            else:
                refined_entries.append(current_entry)
                current_entry = next_entry.copy()
                current_start = next_start
                current_end = next_end
        refined_entries.append(current_entry)
        
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(refined_entries, f, indent=4)
    print(f"[SUCCESS] Pure Python refined timeline saved to: {output_json_path}")
except Exception as e:
    print(f"[ERROR] Pure Python refinement failed: {e}")